Classification STEPS:
LOGS : Directly Target Label and Message ko embedding karke training nahi

USED Clustering along with embedding distance
1- REGEX Pattern wale find -> REGEX Label
2- Non Regex ->
  - a - Fix sopurce alag kar diye uska target
  - b - Not fixed source -> Embedding and train logistic se

Companies Logs ->
- Multiple Sources(Billing, HR, Developer)
- Categories Criticality: Critical, Security, HTTP Status: UserAction

- Filter Regex: Patterns Find in Logs

In [2]:
import pandas as pd

df = pd.read_csv("/content/synthetic_logs.csv")
df.head()

,timestamp,source,log_message,target_label,complexity
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert


In [3]:
df.shape

(2410, 5)

In [4]:
df.source.unique()

array(['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem',
       'ThirdPartyAPI', 'LegacyCRM'], dtype=object)

In [5]:
df.target_label.unique()

array(['HTTP Status', 'Critical Error', 'Security Alert', 'Error',
       'System Notification', 'Resource Usage', 'User Action',
       'Workflow Error', 'Deprecation Warning'], dtype=object)

In [6]:
df[df.target_label=='System Notification'].sample(10)

,timestamp,source,log_message,target_label,complexity
1197,5/28/2025 15:30,ThirdPartyAPI,System updated to version 4.9.9.,System Notification,regex
176,2/12/2025 10:17,ModernCRM,System updated to version 5.5.4.,System Notification,regex
1663,10/27/2025 22:04,AnalyticsEngine,System reboot initiated by user User315.,System Notification,regex
972,3/21/2025 23:04,AnalyticsEngine,File data_1124.csv uploaded successfully by us...,System Notification,regex
885,11/11/2025 11:45,BillingSystem,System updated to version 5.8.1.,System Notification,regex
1946,4/1/2025 15:27,ThirdPartyAPI,Backup started at 2025-02-07 07:50:02.,System Notification,regex
15,5/1/2025 9:41,ModernCRM,Backup completed successfully.,System Notification,regex
980,4/20/2025 18:10,AnalyticsEngine,Backup started at 2025-05-06 07:03:31.,System Notification,regex
194,8/23/2025 1:02,ModernHR,File data_1503.csv uploaded successfully by us...,System Notification,regex
852,3/31/2025 5:20,ModernCRM,System reboot initiated by user User811.,System Notification,regex


In [7]:
df[df.log_message.str.startswith("System reboot initiated by user")].sample(5)

,timestamp,source,log_message,target_label,complexity
865,2/25/2025 1:40,AnalyticsEngine,System reboot initiated by user User964.,System Notification,regex
2360,5/1/2025 4:21,ThirdPartyAPI,System reboot initiated by user User876.,System Notification,regex
36,11/19/2025 13:14,BillingSystem,System reboot initiated by user User243.,System Notification,regex
2317,3/7/2025 5:44,BillingSystem,System reboot initiated by user User724.,System Notification,regex
693,7/6/2025 21:40,BillingSystem,System reboot initiated by user User159.,System Notification,regex


# Clustering and Embedding

- Directly Sentence Log -> Embedding train nahi
- Need to Regex It and then train the model for categories

- For Regex need to find pattern-
  - Using DBSCAN And Sentence transformer for pass wale vectors

## DB Scan
- Cluster log messages different categories
- Cluster using vectors generated from sentence transformer
- Cosine distnace type man lo
- Categories cluster hue vectors converted now use distance for classificaiton

##  Sentence Transformer -> Pretrained models
- - Embedding  Sentencetransformer BERT hai etc of log and then clusterx them togeter

In [8]:
from sklearn.cluster import DBSCAN
from sentence_transformers import SentenceTransformer

In [9]:
model = SentenceTransformer('all-MiniLM-L6-v2')  # Lightweight embedding model
embeddings = model.encode(df['log_message'].tolist())

In [10]:
embeddings[:5]

array([[-0.1029396 ,  0.03354595, -0.02202608, ...,  0.00457793,
        -0.04259716,  0.00322623],
       [ 0.00804573, -0.03573924,  0.04938739, ...,  0.01538321,
        -0.06230951, -0.02774664],
       [-0.00908221,  0.13003927, -0.05275569, ...,  0.02014104,
        -0.05117096, -0.02930295],
       [-0.09751045,  0.04911302, -0.03977428, ...,  0.02477501,
        -0.03546082, -0.00018599],
       [-0.10468338,  0.05926034, -0.02488501, ...,  0.02502053,
        -0.03719296, -0.02568914]], dtype=float32)

Cluster Baat ke then Finding pattern inside that cluster
- Log Messages -> Convert simple sentences
 - ex user 609 logged in out -> user action

In [11]:
clustering = DBSCAN(eps=0.2, min_samples=1, metric='cosine').fit(embeddings)
# Vectors se clusters create usign cosine distance
df['cluster'] = clustering.labels_

In [12]:
df.head()

,timestamp,source,log_message,target_label,complexity,cluster
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0


In [13]:
# Group by cluster to inspect patterns
clusters = df.groupby('cluster')['log_message'].apply(list)

In [14]:
clusters
# Each cluster with its values in array

,log_message
cluster,
0,[nova.osapi_compute.wsgi.server [req-b9718cd8-...
1,[Email service experiencing issues with sendin...
2,"[Unauthorized access to data was attempted, An..."
3,"[Shard 6 replication task ended in failure, Da..."
4,[File data_6169.csv uploaded successfully by u...
...,...
131,[Global settings have been compromised]
132,[Admin rights elevated for user 1776]
133,[Task assignment for TeamID 3425 could not com...


In [15]:
sorted_clusters = clusters.sort_values(key=lambda x: x.map(len), ascending=False)
#  sort the cluster array with values length descending order

In [16]:
print("Clustered Patterns:")
for cluster_id, messages in sorted_clusters.items():
    if len(messages) > 10:
        print(f"Cluster {cluster_id}:")
        for msg in messages[:5]:
            print(f"  {msg}")

Clustered Patterns:
Cluster 0:
  nova.osapi_compute.wsgi.server [req-b9718cd8-f65e-49cc-8349-6cf7122af137 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" status: 200 len: 1893 time: 0.2675118
  nova.osapi_compute.wsgi.server [req-4895c258-b2f8-488f-a2a3-4fae63982e48 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" HTTP status code -  200 len: 211 time: 0.0968180
  nova.osapi_compute.wsgi.server [req-ee8bc8ba-9265-4280-9215-dbe000a41209 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" RCODE  200 len: 1874 time: 0.2280791
  nova.osapi_compute.wsgi.server [req-f0bffbc3-5ab0-4916-91c1-0a61dd7d4ec2 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2

## Classification Stage 1: Regex
- Pattern match kiya us clusters se unko aise classify

In [17]:
import re
def classify_with_regex(log_message):
    regex_patterns = {
        r"User User\d+ logged (in|out).": "User Action",
        r"Backup (started|ended) at .*": "System Notification",
        r"Backup completed successfully.": "System Notification",
        r"System updated to version .*": "System Notification",
        r"File .* uploaded successfully by user .*": "System Notification",
        r"Disk cleanup completed successfully.": "System Notification",
        r"System reboot initiated by user .*": "System Notification",
        r"Account with ID .* created by .*": "User Action"
    }
    for pattern, label in regex_patterns.items():
        if re.search(pattern, log_message):
            return label
    return None

In [18]:
classify_with_regex("User User123 logged in.")

'User Action'

In [19]:
# New column create aise classified
# Apply regex classification
df['regex_label'] = df['log_message'].apply(lambda x: classify_with_regex(x))
df[df['regex_label'].notnull()]

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
7,10/11/2025 8:44,ModernHR,File data_6169.csv uploaded successfully by us...,System Notification,regex,4,System Notification
14,1/4/2025 1:43,ThirdPartyAPI,File data_3847.csv uploaded successfully by us...,System Notification,regex,4,System Notification
15,5/1/2025 9:41,ModernCRM,Backup completed successfully.,System Notification,regex,8,System Notification
18,2/22/2025 17:49,ModernCRM,Account with ID 5351 created by User634.,User Action,regex,9,User Action
27,9/24/2025 19:57,ThirdPartyAPI,User User685 logged out.,User Action,regex,11,User Action
...,...,...,...,...,...,...,...
2376,6/27/2025 8:47,ModernCRM,System updated to version 2.0.5.,System Notification,regex,21,System Notification
2381,9/5/2025 6:39,ThirdPartyAPI,Disk cleanup completed successfully.,System Notification,regex,32,System Notification
2394,4/3/2025 13:13,ModernHR,Disk cleanup completed successfully.,System Notification,regex,32,System Notification
2395,5/2/2025 14:29,ThirdPartyAPI,Backup ended at 2025-05-06 11:23:16.,System Notification,regex,13,System Notification


In [20]:
df[df['regex_label'].isnull()].head(5)
# REGEX Pattern nahi match hua

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0,None
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1,None
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2,None
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0,None
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0,None


## Classification Stage 2: Classification Using Embeddings

REGEX NAHI MATCH
- ['Workflow Error', 'Deprecation Warning'] TARGET From LegacyCRM

In [21]:
df_non_regex = df[df['regex_label'].isnull()].copy()
df_non_regex.shape

(1910, 7)

In [22]:
df_legacy = df_non_regex[df_non_regex.source=="LegacyCRM"]
df_legacy.target_label.unique()

array(['Workflow Error', 'Deprecation Warning'], dtype=object)

Remaining not legacy walo ko classify using BERT

In [23]:
df_non_legacy = df_non_regex[df_non_regex.source!="LegacyCRM"]
df_non_legacy.target_label.unique()
# ['HTTP Status', 'Critical Error', 'Security Alert', 'Error',
#  'Resource Usage'
#  Inka classification banana hai

array(['HTTP Status', 'Critical Error', 'Security Alert', 'Error',
       'Resource Usage'], dtype=object)

In [24]:
df_non_legacy.shape

(1903, 7)

In [25]:
#  Sentence Embeddin
model = SentenceTransformer('all-MiniLM-L6-v2')  # Lightweight embedding model
embeddings_filtered = model.encode(df_non_legacy['log_message'].tolist())

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [26]:
len(embeddings_filtered)

1903

In [27]:
embeddings_filtered[1].shape
# vector of that sentence

(384,)

In [28]:
X = embeddings_filtered
y = df_non_legacy['target_label'].values

In [29]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
report = classification_report(y_test, y_pred)
print(report)

                precision    recall  f1-score   support

Critical Error       0.91      1.00      0.95        48
         Error       0.98      0.89      0.93        47
   HTTP Status       1.00      1.00      1.00       304
Resource Usage       1.00      1.00      1.00        49
Security Alert       1.00      0.99      1.00       123

      accuracy                           0.99       571
     macro avg       0.98      0.98      0.98       571
  weighted avg       0.99      0.99      0.99       571



SKLEARN Models Export

In [ ]:
y_pred[0]
X_test[0]

In [34]:
import joblib
joblib.dump(clf, '/content/models/log_classifier.joblib')

['/content/models/log_classifier.joblib']